# Interpreting a Small Medical Regression: Diabetes Progression

**Family:** End-to-end project | **Dataset:** `diabetes` | **Task:** regression

Which baseline measurements relate to progression, how stable is that answer, and what does a sparse model keep?

## Table of Contents

- 1. Problem statement
- 2. Business or research context
- 3. Dataset documentation
- 4. Research references
- 5. Exploratory analysis
- 6. Preprocessing
- 7. Model selection
- 8. Model training
- 9. Evaluation
- 10. Comparison
- 11. Interpretation
- 12. Limitations
- 13. Final recommendations

## 1. Problem statement

Predict disease progression one year ahead and identify measurements that matter, with a stability check.

## 2. Business or research context

In a small clinical sample, an interpretable model is more useful than a black box. But feature rankings from 442 rows can change a lot between resamples, so we test that.

## 3. Dataset documentation

**Diabetes progression** from Efron, Hastie, Johnstone and Tibshirani (2004), shipped with scikit-learn. 442 patients, 10 baseline features (`age`, `sex`, `bmi`, `bp`, and six blood serum measures `s1` to `s6`), target is a quantitative measure of disease progression one year after baseline. The provider centred and scaled the features. Data terms of the original source are not stated by the provider. Limits: small sample, no time dimension, `sex` is a standardised number.

## 4. Research references

Verification status is from the build environment on 2026-11-17. Entries marked *not verified* must be checked before citing.

- Efron, B., Hastie, T., Johnstone, I. and Tibshirani, R. (2004). *Least angle regression*. The Annals of Statistics 32(2):407-499. https://doi.org/10.1214/009053604000000067 Verification: Crossref 2026-11-17
- Tibshirani, R. (1996). *Regression Shrinkage and Selection Via the Lasso*. Journal of the Royal Statistical Society B 58(1):267-288. https://doi.org/10.1111/j.2517-6161.1996.tb02080.x Verification: Crossref 2026-11-17
- Zou, H. and Hastie, T. (2005). *Regularization and Variable Selection Via the Elastic Net*. Journal of the Royal Statistical Society B 67(2):301-320. https://doi.org/10.1111/j.1467-9868.2005.00503.x Verification: Crossref 2026-11-17

The data and the lasso path come from the LARS paper (Efron et al.). Lasso (Tibshirani) and elastic net (Zou and Hastie) are compared.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_score
from ml_lab.data import load_dataset
from ml_lab.preprocessing import split_three_way, make_model_pipeline, build_preprocessor
from ml_lab.evaluation import evaluate_estimator, internal_scores, external_scores
from ml_lab.experiments import ExperimentRecord, save_figure
from ml_lab.utils import set_seed, print_environment
from ml_lab import visualization as viz
PROJECT_ID = "diabetes_interpretation"
SEED = 42
set_seed(SEED)
viz.apply_style()
print_environment()

python 3.13.18 | Linux-6.18.44-fc-v595-x86_64-with-glibc2.39
  numpy==2.5.3
  pandas==3.0.5
  scipy==1.18.1
  scikit-learn==1.9.1
  matplotlib==3.11.2
  xgboost==3.4.1
  lightgbm==4.7.0
  catboost==1.2.10
  umap-learn==0.6.0
  torch==2.15.0
  gymnasium==1.4.0


In [2]:
bundle = load_dataset("diabetes")
X, y = bundle.X, bundle.y
Xtr, Xva, Xte, ytr, yva, yte = split_three_way(X, y, task="regression", seed=SEED)
print(len(Xtr), len(Xva), len(Xte))

264 89 89


## 5. Exploratory analysis

Correlation between the six serum measures.

In [3]:
display(X.corr().round(2))

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
age,1.00,0.17,0.19,0.34,0.26,0.22,-0.08,0.20,0.27,0.30
sex,0.17,1.00,0.09,0.24,0.04,0.14,-0.38,0.33,0.15,0.21
bmi,0.19,0.09,1.00,0.40,0.25,0.26,-0.37,0.41,0.45,0.39
bp,0.34,0.24,0.40,1.00,0.24,0.19,-0.18,0.26,0.39,0.39
s1,0.26,0.04,0.25,0.24,1.00,0.90,0.05,0.54,0.52,0.33
s2,0.22,0.14,0.26,0.19,0.90,1.00,-0.20,0.66,0.32,0.29
s3,-0.08,-0.38,-0.37,-0.18,0.05,-0.20,1.00,-0.74,-0.40,-0.27
s4,0.20,0.33,0.41,0.26,0.54,0.66,-0.74,1.00,0.62,0.42
s5,0.27,0.15,0.45,0.39,0.52,0.32,-0.40,0.62,1.00,0.46
s6,0.30,0.21,0.39,0.39,0.33,0.29,-0.27,0.42,0.46,1.00


## 6. Preprocessing

Features are already centred and scaled by the data provider. No further processing.

In [4]:
print("feature means (should be about 0):", np.round(X.mean().values[:3], 3))

feature means (should be about 0): [-0.  0. -0.]


## 7. Model selection

Linear regression, lasso, elastic net and gradient boosting. Penalty strengths are chosen on the validation rows.

In [5]:
from sklearn.linear_model import LinearRegression, Lasso, ElasticNet
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
rmse = lambda a, b: float(mean_squared_error(a, b) ** 0.5)  # noqa: E731
la = min([0.01, 0.03, 0.1, 0.3, 1.0], key=lambda a: rmse(yva, Lasso(alpha=a).fit(Xtr, ytr).predict(Xva)))
ea = min([0.01, 0.03, 0.1, 0.3, 1.0], key=lambda a: rmse(yva, ElasticNet(alpha=a, l1_ratio=0.5).fit(Xtr, ytr).predict(Xva)))
cands = {"dummy": DummyRegressor(), "linear": LinearRegression(), "lasso": Lasso(alpha=la), "elastic_net": ElasticNet(alpha=ea, l1_ratio=0.5),
         "gradient_boosting": GradientBoostingRegressor(n_estimators=150, max_depth=2, learning_rate=0.05, random_state=SEED)}
fitted = {n: m.fit(Xtr, ytr) for n, m in cands.items()}
sel = pd.Series({n: rmse(yva, f.predict(Xva)) for n, f in fitted.items()}, name="validation RMSE")
display(sel.round(2).to_frame()); BEST = sel.drop("dummy").idxmin(); print("selected:", BEST)

,validation RMSE
dummy,75.64
linear,51.19
lasso,51.02
elastic_net,57.43
gradient_boosting,53.04


selected: lasso


## 8. Model training

Refit on training plus validation rows.

In [6]:
Xtv, ytv = pd.concat([Xtr, Xva]), pd.concat([ytr, yva])
final = clone(cands[BEST]).fit(Xtv, ytv)
cv = cross_val_score(clone(cands[BEST]), Xtv, ytv, cv=KFold(5, shuffle=True, random_state=SEED), scoring="neg_root_mean_squared_error")
print(f"CV RMSE {-cv.mean():.2f} +- {cv.std(ddof=1):.2f}")

CV RMSE 55.21 +- 3.92


## 9. Evaluation

Test set, scored once, with baseline.

In [7]:
m_test = evaluate_estimator(final, Xte, yte, "regression"); m_base = evaluate_estimator(fitted["dummy"], Xte, yte, "regression")
METRICS = {**{f"test_{k}": v for k, v in m_test.items()}, "baseline_RMSE": m_base["RMSE"], "cv_RMSE_mean": float(-cv.mean()), "cv_RMSE_std": float(cv.std(ddof=1))}
CONFIG = {"selected_model": BEST, "lasso_alpha": la, "elastic_net_alpha": ea}
EXTRA = {"validation_RMSE": sel.round(3).to_dict()}
ALGORITHM = BEST
display(pd.DataFrame({"model": m_test, "dummy": m_base}).round(3))

record = ExperimentRecord(notebook=PROJECT_ID, family="project", algorithm=ALGORITHM, dataset="diabetes", task="regression", seed=SEED, config=CONFIG, metrics=METRICS, extra=EXTRA)
print("saved:", record.save())
display(pd.Series(METRICS, name="metric").round(4).to_frame())

,model,dummy
MAE,42.854,63.531
MSE,2798.193,5314.436
RMSE,52.898,72.900
R2,0.472,-0.003
MAPE_percent,37.629,60.880


saved: experiments/results/diabetes_interpretation.json


,metric
test_MAE,42.8544
test_MSE,2798.1935
test_RMSE,52.8980
test_R2,0.4719
test_MAPE_percent,37.6286
baseline_RMSE,72.9002
cv_RMSE_mean,55.2053
cv_RMSE_std,3.9210


## 10. Comparison

All models on the test rows, for comparison only.

In [8]:
display(pd.Series({n: rmse(yte, f.predict(Xte)) for n, f in fitted.items()}, name="test RMSE").round(2).to_frame())

,test RMSE
dummy,72.90
linear,54.16
lasso,52.95
elastic_net,57.66
gradient_boosting,53.10


## 11. Interpretation

Lasso coefficients and selection frequency across 200 bootstrap resamples of the training plus validation rows.

In [9]:
coef = pd.Series(Lasso(alpha=la).fit(Xtv, ytv).coef_, index=X.columns)
rng = np.random.default_rng(SEED); picked = np.zeros(X.shape[1])
for _ in range(200):
    i = rng.integers(0, len(Xtv), len(Xtv))
    picked += (Lasso(alpha=la).fit(Xtv.iloc[i], ytv.iloc[i]).coef_ != 0)
display(pd.DataFrame({"lasso_coef": coef, "selected_in_share_of_bootstraps": picked / 200}).round(2).sort_values("lasso_coef", key=abs, ascending=False))

,lasso_coef,selected_in_share_of_bootstraps
bmi,552.70,1.00
s5,447.92,1.00
bp,303.37,1.00
s3,-229.26,0.98
sex,-152.66,0.98
s1,-81.37,0.73
s6,29.64,0.68
age,0.00,0.42
s2,-0.00,0.23
s4,0.00,0.32


## 12. Limitations

442 patients, one study, features pre-scaled by the provider, and `sex` is a standardised number. Associations, not causes. Rankings of correlated serum measures are unstable, as the bootstrap table shows.

## 13. Final recommendations

Report sparse model results with selection frequencies, not a single ranking. Use these results to generate hypotheses and test them on new patient data.